## Previous code setup code needed
code from the previous milestone that I have already done but need for main model: Data loading, training/test split, forward chaining cv folds & the regression metrics I'll be using.

In [1]:
#Data loading
import sys, platform
print(platform.python_version())
print(sys.executable)   # this should point inside your .venv

FEATURES = ["AT","AP","AH","AFDP","GTEP","TIT","TAT","TEY","CDP"]
TARGET = "NOX"
assert "year" not in FEATURES and "CO" not in FEATURES

from pathlib import Path
import pandas as pd

DATA_DIR = Path("..") / "data"

frames = []
for year in range(2011, 2016):
    year_df = pd.read_csv(DATA_DIR / f"gt_{year}.csv")
    year_df["year"] = year
    frames.append(year_df)

df = pd.concat(frames, ignore_index=True)
print(df.shape)
print(df.groupby("year").size())

# train/test split

train = df[df.year <= 2013]
test  = df[df.year >= 2014]
X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]
print(len(train), len(test))

#forward-chaining CV folds

folds = [((2011,), 2012), ((2011, 2012), 2013)]   # (train years, validation year)
for train_years, val_year in folds:
    num_of_train_rows = df.year.isin(train_years).sum()
    num_of_val_rows = (df.year == val_year).sum()
    print(train_years, "->", val_year, num_of_train_rows, num_of_val_rows)

#regression metrics
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def regression_metrics(y_true, y_pred):
    """Compute regression metrics for NOx predictions.

    Returns: dict with MAE and RMSE (mg/m3), R2, and MeanSignedError
        (mean of prediction - actual; positive = overprediction).
    Assumes: y_true and y_pred are equal-length 1-D numeric arrays.
    Fails: raises an error on mismatched lengths, NaNs or empty input.
        R2 is measured against the mean of y_true (the evaluation set),
        so a constant prediction can score below 0.
    """
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MeanSignedError": float(np.mean(np.asarray(y_pred) - np.asarray(y_true))),
    }

3.14.7
C:\Users\omars\Documents\GitHub\CT4101-ML\.venv\Scripts\python.exe
(36733, 12)
year
2011    7411
2012    7628
2013    7152
2014    7158
2015    7384
dtype: int64
22191 14542
(2011,) -> 2012 7411 7628
(2011, 2012) -> 2013 15039 7152


## Gradient Boosting Model

In [2]:
from itertools import product
from time import perf_counter
from sklearn.ensemble import HistGradientBoostingRegressor

GB_GRID = {
    # chosen alternative settings and default sickitlearn settings
    "learning_rate": [0.05, 0.1],
    "max_iter": [100, 300],
    "max_leaf_nodes": [15, 31],
}

grid_rows, grid_fold_metrics = [], []
for learning_rate, max_iter, max_leaf_nodes in product(*GB_GRID.values()):
    params = {"learning_rate": learning_rate, "max_iter": max_iter, "max_leaf_nodes": max_leaf_nodes}
    # train seconds needed for comparison with neural network later
    fold_metrics, train_seconds = [], 0.0

    for train_years, val_year in folds:
        train_fold = df[df.year.isin(train_years)]
        val_fold = df[df.year == val_year]

        # raw features, no scaler; early stopping off so no random validation split
        model = HistGradientBoostingRegressor(early_stopping=False, **params)
        start = perf_counter()
        model.fit(train_fold[FEATURES], train_fold[TARGET])
        train_seconds += perf_counter() - start

       # predicts NOx for the validation year and store this fold's scores
        fold_metrics.append(regression_metrics(val_fold[TARGET], model.predict(val_fold[FEATURES])))

    # after both folds: saves full scores and a summary row for this combination
    grid_fold_metrics.append(fold_metrics)
    grid_rows.append({
        **params,
        "Fold 1 MAE": fold_metrics[0]["MAE"],
        "Fold 2 MAE": fold_metrics[1]["MAE"],
        "CV MAE": np.mean([metrics["MAE"] for metrics in fold_metrics]),
        "Train time (s)": train_seconds,
    })

# builds the table, picks the lowest CV MAE and shows its settings
grid_df = pd.DataFrame(grid_rows)
best_index = grid_df["CV MAE"].idxmin()
best_gb_params = {name: grid_rows[best_index][name] for name in GB_GRID}
print("Chosen settings:", best_gb_params)
grid_df.sort_values("CV MAE").round(3)

Chosen settings: {'learning_rate': 0.05, 'max_iter': 100, 'max_leaf_nodes': 31}


,learning_rate,max_iter,max_leaf_nodes,Fold 1 MAE,Fold 2 MAE,CV MAE,Train time (s)
1,0.05,100,31,3.824,5.678,4.751,2.931
2,0.05,300,15,3.793,5.715,4.754,40.483
4,0.10,100,15,3.865,5.650,4.758,0.797
0,0.05,100,15,3.862,5.668,4.765,1.603
3,0.05,300,31,3.836,5.723,4.779,8.586
6,0.10,300,15,3.885,5.675,4.780,61.549
5,0.10,100,31,3.864,5.757,4.810,3.103
7,0.10,300,31,3.927,5.757,4.842,13.625


In [3]:
# extra figures for explanation

# average MAE per fold across all 8 settings
print(f"Mean fold 1 MAE: {grid_df['Fold 1 MAE'].mean():.2f}, mean fold 2 MAE: {grid_df['Fold 2 MAE'].mean():.2f}")

# training time of 300 trees vs 100 trees, same learning_rate and max_leaf_nodes
pair_columns = ["learning_rate", "max_leaf_nodes"]
time_100 = grid_df[grid_df.max_iter == 100].set_index(pair_columns)["Train time (s)"]
time_300 = grid_df[grid_df.max_iter == 300].set_index(pair_columns)["Train time (s)"]
print((time_300 / time_100).round(2))

Mean fold 1 MAE: 3.86, mean fold 2 MAE: 5.70
learning_rate  max_leaf_nodes
0.05           15                25.26
               31                 2.93
0.10           15                77.24
               31                 4.39
Name: Train time (s), dtype: float64


I used gradient boosting as it can capture non linear relationships between the sensor features. It's tree based, so it doesn't need scaling and can be trained directly on the raw features. I set early_stopping to False as the default uses a random 10% validation split when the training set has over 10,000 rows. Fold 2's training set has 15,039 rows so early stopping would switch on and hold out a random 10% of the training rows and this would break the chronological split. With these settings the model is deterministic, so no random seed is needed.

I tested a small grid using the scikit-learn defaults and one alternative for each setting: learning_rate [0.05, 0.1], max_iter [100, 300] and max_leaf_nodes [15, 31]. This gives 8 combinations. I compared them using the average MAE across the two forward chaining folds only (train 2011, validate 2012; train 2011-2012, validate 2013). The 2014-2015 test years were not used for tuning.

The best settings were learning_rate = 0.05, max_iter = 100 and max_leaf_nodes = 31, with a CV MAE of 4.75 mg/m³ (fold 1: 3.82, fold 2: 5.68). This is about 42% (8.15 - 4.75 = 3.40, 3.40 / 8.15 = 0.417 = 41.7% = 42%) lower than the baseline MAE of 8.15. Training both folds took 2.8 seconds. However, all 8 combinations scored very similarly, between 4.75 and 4.84, a difference of only 0.09 mg/m³. The scikit-learn defaults (0.1, 100, 31) scored 4.81, so there isn't much difference. Increasing max_iter to 300 was worse in 3 of the 4 pairs, only slightly better in the other (4.754 vs 4.765), and took about 2–6 times longer to train. All settings also had a higher MAE on fold 2 (around 5.7) than fold 1 (around 3.9).

This suggests the model isn't very sensitive to these hyperparameter settings. Adding more trees doesn't seem to help with predicting the next year and mainly takes longer to train. Fold 2 being worse could mean that 2013 was harder to predict, possibly due to NOx rising across 2011-2013. One limitation is that I used the same two folds to choose the settings and report the CV score, so the MAE of 4.75 may be slightly optimistic. The small differences between settings may also just be specific to these folds and might not carry over to new data.

## Neural Network

In [4]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# the two network sizes being compared
MLP_SETTINGS = {
    "Network A (32,)": (32,),
    "Network B (64, 32)": (64, 32),
}

# same for both networks, scikit-learn defaults written out so they're recorded
MLP_FIXED = {
    "activation": "relu",
    "solver": "adam",
    "batch_size": 200,
    "learning_rate_init": 0.001,
    "alpha": 0.0001,             # L2 penalty
    "max_iter": 500,             # max epochs
    "tol": 1e-4,                 # stops when training loss improves by less than this...
    "n_iter_no_change": 10,      # ...for 10 epochs in a row
    "early_stopping": False,     # no random validation split
    "random_state": RANDOM_STATE,
}

mlp_rows, mlp_fold_metrics = [], {}
for setting_name, hidden_layer_sizes in MLP_SETTINGS.items():
    # train seconds needed for comparison with gradient boosting
    fold_metrics, train_seconds, epochs_run = [], 0.0, []

    for train_years, val_year in folds:
        train_fold = df[df.year.isin(train_years)]
        val_fold = df[df.year == val_year]

        # scaler inside the pipeline so it's only fitted on each fold's training years
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", MLPRegressor(hidden_layer_sizes=hidden_layer_sizes, **MLP_FIXED)),
        ])
        start = perf_counter()
        model.fit(train_fold[FEATURES], train_fold[TARGET])
        train_seconds += perf_counter() - start

        # epochs actually run, 500 means it hit the cap instead of converging
        epochs_run.append(model.named_steps["mlp"].n_iter_)
        # predicts NOx for the validation year and stores this fold's scores
        fold_metrics.append(regression_metrics(val_fold[TARGET], model.predict(val_fold[FEATURES])))

    # after both folds: saves full scores and a summary row for this network
    mlp_fold_metrics[setting_name] = fold_metrics
    mlp_rows.append({
        "Setting": setting_name,
        "Fold 1 MAE": fold_metrics[0]["MAE"],
        "Fold 2 MAE": fold_metrics[1]["MAE"],
        "CV MAE": np.mean([metrics["MAE"] for metrics in fold_metrics]),
        "Epochs (fold 1, fold 2)": tuple(epochs_run),
        "Train time (s)": train_seconds,
    })

# builds the table, picks the lowest CV MAE and shows the chosen network
mlp_df = pd.DataFrame(mlp_rows).set_index("Setting")
best_mlp_name = mlp_df["CV MAE"].idxmin()
print("Chosen network:", best_mlp_name)
mlp_df.round(3)

Chosen network: Network A (32,)


,Fold 1 MAE,Fold 2 MAE,CV MAE,"Epochs (fold 1, fold 2)",Train time (s)
Setting,,,,,
"Network A (32,)",4.133,5.359,4.746,"(466, 452)",29.277
"Network B (64, 32)",4.472,5.581,5.027,"(470, 271)",35.678
